# 02 — Several retrieval mixtures at each position

A single head uses one source distribution for all its value coordinates. Multiple heads permit several distributions; they do not split the sentence into separate token ranges.

For $X:[B,T,D]$, each projected tensor is reshaped to $[B,H,T,d]$. Each head computes $O_h=\mathrm{softmax}(Q_hK_h^\top/\sqrt d+M)V_h$. Concatenate on the feature axis, then apply $W_O$. This output projection is not the vocabulary head.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.](../figures/chapter-05/day-05-02_multi_head_attention-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='attention', modern=False))

## Follow Q, K, and V through one attention layer

Q and K create routing probabilities after scaling and causal masking. V bypasses softmax and supplies content; concatenation and W_O return the update to model width.

![Follow Q, K, and V through one attention layer. Q and K create routing probabilities after scaling and causal masking. V bypasses softmax and supplies content; concatenation and W_O return the update to model width.](../figures/chapter-05/day-05-02_multi_head_attention-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

Projection matrix shapes use the mathematical row-vector convention `X @ W`; PyTorch `Linear.weight` stores the transpose of this displayed matrix.


In [ ]:
show_visual(architecture.attention_detail())

## 1. Implement the head split

Project one shared input into Q/K/V, then split heads. Which axis represents time, and which represents a retrieval head?

**Your prediction:** _Write it here before running the reference._

In [ ]:
cfg = DecoderConfig()
mha = MultiHeadAttention(cfg).double()
x = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
# Your implementation: q = ...; k = ...; v = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
def split(proj):
    return proj(x).reshape(2, 6, 4, 4).transpose(1, 2)
q, k, v = split(mha.q), split(mha.k), split(mha.v)
print("X:", x.shape, "Q/K/V:", q.shape, k.shape, v.shape)
print(inspect.getsource(MultiHeadAttention.forward))

### Why this works

The transpose changes [batch,time,head,feature] to [batch,head,time,feature]. It must be reversed before head features are flattened. Merely reshaping a transposed tensor to the desired output size can silently mix axes.

### Visual explanation — See what splitting into heads preserves

Every panel still contains every token position. Only the learned feature view changes. These are random, untrained query vectors—not linguistic head roles.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See what splitting into heads preserves. Every panel still contains every token position. Only the learned feature view changes. These are random, untrained query vectors—not linguistic head roles.](../figures/chapter-05/day-05-02_multi_head_attention-visual-head-split.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.heads(q[0]))

## 2. Compare independent heads with vectorized attention

Compute each head in a loop, then compare outputs AND input gradients with the vectorized module using identical weights.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Your implementation: per_head = [...]; merged = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
per_head = [attend(q[:, h:h+1], k[:, h:h+1], v[:, h:h+1])[0]
            for h in range(cfg.heads)]
mixed = torch.cat(per_head, dim=1)
merged = mixed.transpose(1, 2).contiguous().reshape(2, 6, 16)
manual = mha.out(merged)
actual, _, weights = mha(x)
close(manual, actual)
probe = torch.randn_like(actual)
ga = torch.autograd.grad((manual*probe).sum(), x, retain_graph=True)[0]
gb = torch.autograd.grad((actual*probe).sum(), x, retain_graph=True)[0]
close(ga, gb)
print("Output error:", float((manual-actual).abs().max().detach()))
print("Gradient error:", float((ga-gb).abs().max()))
print("Head source distributions at the last position:", weights[0, :, -1].detach())

### Why this works

Both forms implement the same operation. Vectorization changes the computation layout, not the learning rule or the tokens a head may access.

### Visual explanation — Read the attention maps

Rows are receiving/query positions; columns are source/key positions. Grey cells marked × are forbidden futures. The outlined row is the last receiver; change selected below to inspect another row.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Read the attention maps. Rows are receiving/query positions; columns are source/key positions. Grey cells marked × are forbidden futures. The outlined row is the last receiver; change selected below to inspect another row.](../figures/chapter-05/day-05-02_multi_head_attention-visual-attention-maps.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.attention_maps(weights[0], selected=5))

### Visual explanation — Watch routing weights become retrieved content

For head 0 at the final position, each probability multiplies a source value vector. Sum down the contribution matrix to obtain the output row. Negative feature values are valid.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Watch routing weights become retrieved content. For head 0 at the final position, each probability multiplies a source value vector. Sum down the contribution matrix to obtain the output row. Negative feature values are valid.](../figures/chapter-05/day-05-02_multi_head_attention-visual-value-mixture.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.value_mixture(weights[0, 0, -1], v[0, 0]))

## 3. Ablate a head and perturb the future

Zero one head's retrieved content without changing its attention probabilities. Separately change future inputs. Which outputs are allowed to change?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict both interventions before running the reference.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
ablated = mixed.clone()
ablated[:, 0] = 0
ablated_output = mha.out(ablated.transpose(1, 2).contiguous().reshape(2, 6, 16))
assert not torch.allclose(ablated_output, actual)
changed = x.detach().clone(); changed[:, 4:] += 3
changed_output = mha(changed)[0]
close(changed_output[:, :4], actual[:, :4])
assert weights.triu(1).abs().max() == 0
print("Head-zeroing output change:", float((ablated_output-actual).norm().detach()))
print("Earlier-position future perturbation error:",
      float((changed_output[:, :4]-actual[:, :4]).abs().max().detach()))

### Why this works

Zeroing a value mixture removes its contribution before W_O. Causal masking separately protects earlier outputs. Neither an attention map nor this random head ablation establishes a fixed human-readable head role.

### Visual explanation — Compare the output before and after removing one head

All panels share a signed scale. The difference is measured after the output projection. Attention probabilities alone cannot show the lost value content.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Compare the output before and after removing one head. All panels share a signed scale. The difference is measured after the output projection. Attention probabilities alone cannot show the lost value content.](../figures/chapter-05/day-05-02_multi_head_attention-visual-head-ablation.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.matrices([actual[0], ablated_output[0], (ablated_output-actual)[0]], ['All heads', 'Head 0 removed', 'Change'], 'Ablating one head changes the projected update'))

## Takeaway and evidence boundary

Next: attention has produced an update of width D. The residual connection determines how this update joins the existing state.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.